# Week 3: Feature Engineering & Signal Discovery
## Fraud Risk Analytics & Detection System

> **Investigative Principles (Week 1 Audit Alignment):**
> 1. **Zero Redundancy:** $D1$ and $C1$ were discovered to be identical ($|r|=1.000$) to hand-built time-since-last and velocity features. We skip duplicate implementations and use $C1$ and $D1$ directly.
> 2. **Zero Temporal Leakage:** All group statistics (amount z-scores) and frequency encoding tables are fit **strictly** on the training partition ($TransactionDT \le 12,192,854$). The test partition ($TransactionDT > 12,192,854$) is transformed using frozen lookups.
> 3. **Non-Collinear Additive Signal:** Newly engineered features must provide genuine additive predictive power without duplicating existing $V/D/C$ feature blocks.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.engineer import FraudFeaturePipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Environment initialized.")

## 1. Load Parquet Data & Verify Temporal Partition

In [ ]:
data_path = PROJECT_ROOT / "data" / "processed" / "train_merged.parquet"
df = pd.read_parquet(data_path)

CUTOFF_DT = 12192854
train_df = df[df["TransactionDT"] <= CUTOFF_DT].copy().reset_index(drop=True)
test_df = df[df["TransactionDT"] > CUTOFF_DT].copy().reset_index(drop=True)

print(f"Full Dataset:  {len(df):,} rows x {len(df.columns)} columns")
print(f"Train Split:   {len(train_df):,} rows (Fraud Rate: {(train_df['isFraud']==1).mean():.3%})")
print(f"Test Split:    {len(test_df):,} rows (Fraud Rate: {(test_df['isFraud']==1).mean():.3%})")

## 2. Fit Leakage-Free Feature Engineering Pipeline

In [ ]:
pipeline = FraudFeaturePipeline()
pipeline.fit(train_df)

train_feat = pipeline.transform(train_df)
test_feat = pipeline.transform(test_df)

print(f"Engineered {len(pipeline.engineered_feature_names)} new features:")
for feat in pipeline.engineered_feature_names:
    print(f" - {feat}")

## 3. Amount Skewness & Log-Transformation
`TransactionAmt` exhibits extreme right skewness ($0.01 to $30,000+). `log_TransactionAmt` normalizes the variance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(train_feat["TransactionAmt"], bins=50, ax=axes[0], color="royalblue", log_scale=True)
axes[0].set_title("Raw Transaction Amount (Log Scale)")
axes[0].set_xlabel("TransactionAmt ($)")

sns.kdeplot(data=train_feat, x="log_TransactionAmt", hue="isFraud", common_norm=False, ax=axes[1], palette=["#2ecc71", "#e74c3c"])
axes[1].set_title("Log-Transformed Amount: Fraud vs Legit")
axes[1].set_xlabel("log(1 + TransactionAmt)")

plt.tight_layout()
plt.show()

## 4. Amount Z-Score Analysis by Card & Entity
Fraudulent transactions frequently deviate significantly from the typical spending baseline of the specific card (`card1`).

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_feat, x="isFraud", y="amt_zscore_card1", palette=["#2ecc71", "#e74c3c"], showfliers=False)
plt.title("Card-Level Amount Z-Score: Legitimate (0) vs Fraud (1)")
plt.xlabel("Fraud Label (isFraud)")
plt.ylabel("Amount Z-Score (card1 group)")
plt.show()

## 5. Cyclical Temporal Patterns (Diurnal & Weekly Cycles)

In [ ]:
hourly_fraud = train_feat.groupby("hour_of_day")["isFraud"].mean()

plt.figure(figsize=(12, 4))
plt.plot(hourly_fraud.index, hourly_fraud.values * 100, marker="o", color="#e74c3c", linewidth=2.5)
plt.title("Fraud Rate by Hour of Day (24-Hour Diurnal Cycle)")
plt.xlabel("Relative Hour of Day (0 to 23)")
plt.ylabel("Fraud Rate (%)")
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.show()

## 6. Correlation Matrix & Redundancy Verification
Auditing newly engineered features against $isFraud$, $C1$, and $D1$ to ensure zero collinear redundancy ($|r| < 0.85$).

In [ ]:
check_cols = [
    "isFraud", "C1", "D1", "log_TransactionAmt", 
    "amt_zscore_card1", "amt_diff_mean_card1", "amt_ratio_mean_card1",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "freq_card1", "freq_ProductCD", "email_match_flag"
]
corr_matrix = train_feat[check_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-0.5, vmax=0.5)
plt.title("Cross-Feature Correlation Matrix (Additive Signal Verification)")
plt.show()